# Transaction-cost estimation for the SCDLDS Indian-equity universe

This notebook implements two low-frequency effective-cost estimators:

1. **Roll (1984)**: $c=\sqrt{-\operatorname{Cov}(\Delta p_t,\Delta p_{t-1})}$ when the covariance is negative.
2. **Hasbrouck (2009)**: the Bayesian market-factor Roll model
   $\Delta p_t=c(q_t-q_{t-1})+\beta_m r_{m,t}+u_t$, with latent trade direction
   $q_t\in\{-1,+1\}$ (and $q_t=0$ only when a no-trade midpoint is actually observed).

`c` is the **one-way effective cost / half-spread** in log-price units. The full
effective spread is `2*c`. This distinction matters for the website: its turnover
charge should use `c * 10,000` basis points per dollar traded, not the doubled spread.

Primary sources:

- Richard Roll, “A Simple Implicit Measure of the Effective Bid-Ask Spread in an Efficient Market,” *Journal of Finance* 39 (1984), 1127–1139. DOI: https://doi.org/10.1111/j.1540-6261.1984.tb03897.x
- Joel Hasbrouck, “Trading Costs and Returns for U.S. Equities: Estimating Effective Costs from Daily Data,” *Journal of Finance* 64 (2009), 1445–1477. DOI: https://doi.org/10.1111/j.1540-6261.2009.01469.x
- Hasbrouck’s official updated code/data page: https://pages.stern.nyu.edu/~jhasbrou/Research/GibbsCurrent/gibbsCurrentIndex.html
- `Data/Papers/SCDLDS_Documentation.pdf` in this repository for the Indian-market universe and filters.

The notebook deliberately does **not** reuse the older files under
`/home/Samyak.baid_ug2024/transaction_cost/output/results`: that Gibbs code used
`q_t` rather than `q_t-q_{t-1}`, treated zero price changes as known no-trade
midpoints, and did not use the adjacent-residual conditional distribution for `q`.


## Research design and SCDLDS filter policy

The website backtest reads the monthly panel
`Data/Factor_Data/company_month_ALL_FACTOR_LABELS_FINAL_COMPACT.csv`. Its
`(co_code, Month)` membership is therefore the **authoritative filtered universe**.
Using that membership prevents a second implementation from silently drifting from
the website.

The documentation rules are also reconstructed below as an audit:

- use NSE prices where available and BSE otherwise;
- exclude a stock when its median daily closing price in the preceding year is at or below ₹10;
- exclude a stock when September-end market capitalization is below 10% of the cross-sectional median;
- exclude a stock unless it traded at least once in every full trading week during the 12 months before September;
- negative book equity is factor-specific: it excludes value, profitability, and investment assignments, but not size or momentum. The website panel’s blank factor labels enforce this at portfolio-selection time, so it is not imposed globally on transaction-cost estimation.

Estimates are formed over the SCDLDS October–September portfolio year. A deployable
schedule lags the estimate one full portfolio year to prevent look-ahead. The paper
uses annual samples; it explicitly warns against firm-month Gibbs estimates because
short samples are dominated by the (necessarily biased) nonnegative prior.
Any October–September sample not complete at the daily panel's data cutoff is excluded.

The paper's 1,000-sweep/200-burn schedule is the first pass. Any empirical chain with
ESS below 100 or relative MCSE above 5% is automatically rerun for 4,000 sweeps with
800 burn-in observations; this changes simulation length, not the statistical model.

**Data limitation:** the daily panel contains only observed trade-price rows and has
no zero-volume records. Consequently, no observation is asserted to be a CRSP-style
no-trade quote midpoint; all latent directions are sampled from `{-1,+1}`. Missing
calendar days are handled by using cumulative NIFTY 500 log returns between successive
observed stock prices.


In [1]:
from __future__ import annotations

import json
import math
import os
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit, ndtr, ndtri
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/home/Samyak.baid_ug2024/Quantfin-scdlds.com")
DAILY_PANEL_PATH = Path("/home/Samyak.baid_ug2024/transaction_cost/stock_panel.parquet")
NIFTY_DAILY_PATH = Path("/home/Samyak.baid_ug2024/transaction_cost/Nifty 500 Historical Data_daily.csv")
WEBSITE_PANEL_PATH = PROJECT_ROOT / "Data/Factor_Data/company_month_ALL_FACTOR_LABELS_FINAL_COMPACT.csv"
OUTPUT_DIR = PROJECT_ROOT / "notebooks/transaction_cost_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# quick: reproducible stratified sample, suitable for an interactive notebook run.
# full: every eligible firm/portfolio-year (computationally much heavier).
RUN_MODE = "full"
if RUN_MODE not in {"quick", "full"}:
    raise ValueError("TCOST_RUN_MODE must be 'quick' or 'full'")

SEED = 20260721
N_SWEEPS = 1_000
BURN_IN = 200
ADAPTIVE_N_SWEEPS = 4_000
ADAPTIVE_BURN_IN = 800
MIN_CHAIN_ESS = 100.0
MAX_RELATIVE_MCSE = 0.05
MIN_TRADE_PRICES = 60  # Hasbrouck's official updated implementation threshold
QUICK_GROUPS_PER_YEAR = 8
FILTER_AUDIT_YEARS = [2010, 2015, 2020, 2024] if RUN_MODE == "quick" else list(range(2007, 2025))

print({
    "run_mode": RUN_MODE,
    "n_sweeps": N_SWEEPS,
    "burn_in": BURN_IN,
    "min_trade_prices": MIN_TRADE_PRICES,
    "output_dir": str(OUTPUT_DIR),
})


{'run_mode': 'full', 'n_sweeps': 1000, 'burn_in': 200, 'min_trade_prices': 60, 'output_dir': '/home/Samyak.baid_ug2024/Quantfin-scdlds.com/notebooks/transaction_cost_outputs'}


## Load data and establish the authoritative website universe


In [2]:
required = [DAILY_PANEL_PATH, NIFTY_DAILY_PATH, WEBSITE_PANEL_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

t0 = time.time()
raw = pd.read_parquet(
    DAILY_PANEL_PATH,
    columns=["company_code", "company_name", "date", "high", "low", "close", "exchange", "volume", "market_cap"],
)
raw["date"] = pd.to_datetime(raw["date"])
raw = raw.dropna(subset=["company_code", "date", "close"])
raw = raw.loc[raw["close"] > 0].copy()
raw["company_code"] = raw["company_code"].astype("int64")

website = pd.read_csv(WEBSITE_PANEL_PATH, usecols=["co_code", "Month", "lagged_mktcap"])
website["co_code"] = pd.to_numeric(website["co_code"], errors="coerce")
website = website.dropna(subset=["co_code", "Month"]).copy()
website["co_code"] = website["co_code"].astype("int64")
website["Month"] = website["Month"].astype(str).str.slice(0, 7)

def portfolio_year_from_month(month: pd.Series) -> pd.Series:
    year = month.str.slice(0, 4).astype(int)
    mon = month.str.slice(5, 7).astype(int)
    return year.where(mon >= 10, year - 1)

def portfolio_year_from_date(date: pd.Series) -> pd.Series:
    return date.dt.year.where(date.dt.month >= 10, date.dt.year - 1)

website["portfolio_year"] = portfolio_year_from_month(website["Month"])
eligible_pairs = website[["co_code", "portfolio_year"]].drop_duplicates().rename(columns={"co_code": "company_code"})

print(f"Loaded in {time.time()-t0:.1f}s")
print({
    "daily_rows": len(raw),
    "daily_stocks": raw["company_code"].nunique(),
    "daily_dates": (str(raw["date"].min().date()), str(raw["date"].max().date())),
    "website_rows": len(website),
    "website_stocks": website["co_code"].nunique(),
    "website_months": (website["Month"].min(), website["Month"].max()),
    "eligible_stock_portfolio_years": len(eligible_pairs),
})


Loaded in 1.3s
{'daily_rows': 13476477, 'daily_stocks': 5682, 'daily_dates': ('2006-01-02', '2025-12-31'), 'website_rows': 553959, 'website_stocks': 5357, 'website_months': ('2003-10', '2026-05'), 'eligible_stock_portfolio_years': 48027}


## Data-quality checks that affect estimator interpretation


In [3]:
quality = {
    "duplicate_company_date": int(raw.duplicated(["company_code", "date"]).sum()),
    "missing_volume": int(raw["volume"].isna().sum()),
    "zero_volume": int(raw["volume"].eq(0).sum()),
    "negative_volume": int(raw["volume"].lt(0).sum()),
    "missing_market_cap": int(raw["market_cap"].isna().sum()),
    "exchange_counts": raw["exchange"].value_counts(dropna=False).to_dict(),
}
code_exchange_counts = raw.groupby("company_code")["exchange"].nunique()
quality["companies_seen_on_multiple_exchanges_over_time"] = int(code_exchange_counts.gt(1).sum())

# The source already contains at most one exchange per firm/date. Hence it already
# embodies the documented NSE-when-available / BSE-otherwise selection on each date.
assert quality["duplicate_company_date"] == 0
print(json.dumps(quality, indent=2, default=str))

if quality["zero_volume"] == 0:
    print("No zero-volume rows exist: q_t=0 cannot be identified and is never imputed from a zero return.")


{
  "duplicate_company_date": 0,
  "missing_volume": 7341,
  "zero_volume": 0,
  "negative_volume": 0,
  "missing_market_cap": 8938,
  "exchange_counts": {
    "NSE": 7024639,
    "BSE": 6451838
  },
  "companies_seen_on_multiple_exchanges_over_time": 583
}
No zero-volume rows exist: q_t=0 cannot be identified and is never imputed from a zero return.


## Direct reconstruction of the documented filters (audit only)

The exact website membership remains authoritative. This reconstruction is a
diagnostic because the daily panel starts in 2006 (the website starts in 2003),
914 website identifiers have no daily-panel match, and the raw provenance of the
already-selected NSE/BSE series is not available here.

For portfolio year `y` (October `y` through September `y+1`), “the 12 months before
September” is operationalized as September `y-1` through August `y`. A full week is
an ISO Monday–Sunday week wholly inside this window; any exchange holiday week still
counts as a week, and one observed trade during that week is sufficient.


In [4]:
def reconstruct_filters_for_year(raw_panel: pd.DataFrame, y: int) -> tuple[set[int], dict]:
    start = pd.Timestamp(y - 1, 9, 1)
    end = pd.Timestamp(y, 8, 31)
    lookback = raw_panel.loc[raw_panel["date"].between(start, end), ["company_code", "date", "close"]].copy()

    median_price = lookback.groupby("company_code", observed=True)["close"].median()
    passes_penny = set(median_price.index[median_price > 10.0].astype(int))

    week_period = lookback["date"].dt.to_period("W-SUN")
    lookback["week"] = week_period
    full_week_mask = (week_period.dt.start_time >= start) & (week_period.dt.end_time <= end)
    full = lookback.loc[full_week_mask]
    required_weeks = int(full["week"].nunique())
    weeks_by_stock = full.groupby("company_code", observed=True)["week"].nunique()
    passes_liquidity = set(weeks_by_stock.index[weeks_by_stock == required_weeks].astype(int))

    september = raw_panel.loc[
        (raw_panel["date"].dt.year == y) & (raw_panel["date"].dt.month == 9),
        ["company_code", "date", "market_cap"],
    ].dropna(subset=["market_cap"])
    sept_last = september.sort_values(["company_code", "date"]).groupby("company_code", observed=True).tail(1)
    median_mcap = float(sept_last["market_cap"].median()) if len(sept_last) else np.nan
    passes_microcap = set(sept_last.loc[sept_last["market_cap"] >= 0.10 * median_mcap, "company_code"].astype(int))

    reconstructed = passes_penny & passes_liquidity & passes_microcap
    details = {
        "portfolio_year": y,
        "required_full_weeks": required_weeks,
        "price_pass": len(passes_penny),
        "weekly_trade_pass": len(passes_liquidity),
        "microcap_pass": len(passes_microcap),
        "reconstructed_pass": len(reconstructed),
        "september_median_market_cap": median_mcap,
    }
    return reconstructed, details


audit_rows = []
raw_codes = set(raw["company_code"].unique())
for y in FILTER_AUDIT_YEARS:
    reconstructed, details = reconstruct_filters_for_year(raw, y)
    canonical = set(
        eligible_pairs.loc[eligible_pairs["portfolio_year"].eq(y), "company_code"].astype(int)
    ) & raw_codes
    intersection = reconstructed & canonical
    union = reconstructed | canonical
    details.update({
        "canonical_pass_with_daily_match": len(canonical),
        "intersection": len(intersection),
        "precision_vs_canonical": len(intersection) / len(reconstructed) if reconstructed else np.nan,
        "recall_vs_canonical": len(intersection) / len(canonical) if canonical else np.nan,
        "jaccard": len(intersection) / len(union) if union else np.nan,
    })
    audit_rows.append(details)

filter_audit = pd.DataFrame(audit_rows)
filter_audit.to_csv(OUTPUT_DIR / "filter_reconstruction_audit.csv", index=False)
print(filter_audit.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


 portfolio_year  required_full_weeks  price_pass  weekly_trade_pass  microcap_pass  reconstructed_pass  september_median_market_cap  canonical_pass_with_daily_match  intersection  precision_vs_canonical  recall_vs_canonical  jaccard
           2007                   51        1577               1674           1915                1208                     556.2050                             1550          1196                  0.9901               0.7716   0.7657
           2008                   51        1793               1825           2098                1434                     390.5544                             1672          1406                  0.9805               0.8409   0.8271
           2009                   52        1516               1704           2098                1161                     475.8769                             1462          1155                  0.9948               0.7900   0.7868
           2010                   51        1915               2018 

## NIFTY 500 market factor and eligible estimation samples


In [5]:
nifty_raw = pd.read_csv(NIFTY_DAILY_PATH)
nifty_raw["date"] = pd.to_datetime(nifty_raw["Date"], format="%d-%m-%Y", dayfirst=True)
nifty_raw["nifty_close"] = pd.to_numeric(
    nifty_raw["Price"].astype(str).str.replace(",", "", regex=False), errors="coerce"
)
nifty = (
    nifty_raw[["date", "nifty_close"]]
    .dropna()
    .loc[lambda x: x["nifty_close"] > 0]
    .drop_duplicates("date", keep="last")
    .sort_values("date")
)
nifty["market_log_level"] = np.log(nifty["nifty_close"])

raw["portfolio_year"] = portfolio_year_from_date(raw["date"]).astype(int)
daily = raw.merge(eligible_pairs, on=["company_code", "portfolio_year"], how="inner")
daily = daily.merge(nifty[["date", "market_log_level"]], on="date", how="inner")
daily = daily.sort_values(["company_code", "portfolio_year", "date"]).reset_index(drop=True)

group_meta = (
    daily.groupby(["company_code", "portfolio_year"], observed=True)
    .agg(n_prices=("close", "size"), avg_mktcap=("market_cap", "mean"), first_date=("date", "min"), last_date=("date", "max"))
    .reset_index()
)
data_cutoff = daily["date"].max()
latest_completed_portfolio_year = data_cutoff.year - (1 if data_cutoff.month >= 9 else 2)
valid_meta = group_meta.loc[
    (group_meta["n_prices"] >= MIN_TRADE_PRICES)
    & (group_meta["portfolio_year"] <= latest_completed_portfolio_year)
].copy()

def stratified_quick_sample(meta: pd.DataFrame, per_year: int) -> pd.DataFrame:
    chosen = []
    for _, part in meta.groupby("portfolio_year", sort=True):
        part = part.sort_values("avg_mktcap", na_position="last")
        if len(part) <= per_year:
            chosen.append(part)
            continue
        idx = np.linspace(0, len(part) - 1, per_year).round().astype(int)
        chosen.append(part.iloc[np.unique(idx)])
    return pd.concat(chosen, ignore_index=True) if chosen else meta.iloc[0:0]

selected_meta = valid_meta if RUN_MODE == "full" else stratified_quick_sample(valid_meta, QUICK_GROUPS_PER_YEAR)
selected_keys = set(map(tuple, selected_meta[["company_code", "portfolio_year"]].itertuples(index=False, name=None)))

print({
    "nifty_dates": (str(nifty["date"].min().date()), str(nifty["date"].max().date())),
    "eligible_daily_rows": len(daily),
    "valid_stock_portfolio_years": len(valid_meta),
    "daily_data_cutoff": str(data_cutoff.date()),
    "latest_completed_portfolio_year": int(latest_completed_portfolio_year),
    "selected_stock_portfolio_years": len(selected_meta),
    "selected_years": (int(selected_meta["portfolio_year"].min()), int(selected_meta["portfolio_year"].max())),
})


{'nifty_dates': ('2006-01-16', '2026-03-17'), 'eligible_daily_rows': 9188652, 'valid_stock_portfolio_years': 37819, 'daily_data_cutoff': '2025-12-31', 'latest_completed_portfolio_year': 2024, 'selected_stock_portfolio_years': 37819, 'selected_years': (2005, 2024)}


## Correct Roll and Hasbrouck implementations


In [6]:
@dataclass(frozen=True)
class GibbsConfig:
    n_sweeps: int = N_SWEEPS
    burn_in: int = BURN_IN
    prior_c_mean: float = 0.0
    prior_c_var: float = 0.05 ** 2
    prior_beta_mean: float = 1.0
    prior_beta_var: float = 1.0
    prior_sigma_alpha: float = 1e-12
    prior_sigma_scale: float = 1e-12
    initial_sigma2: float = 0.0004


def roll_estimate(delta_p: np.ndarray) -> dict:
    r = np.asarray(delta_p, dtype=float)
    r = r[np.isfinite(r)]
    if len(r) < 3:
        return {"gamma1": np.nan, "c_roll": np.nan, "roll_feasible": False}
    demeaned = r - r.mean()
    gamma1 = float(np.mean(demeaned[1:] * demeaned[:-1]))
    feasible = gamma1 < 0
    return {
        "gamma1": gamma1,
        "c_roll": math.sqrt(-gamma1) if feasible else 0.0,
        "roll_feasible": feasible,
    }


def draw_joint_normal_truncated_c(
    mean: np.ndarray, cov: np.ndarray, rng: np.random.Generator
) -> tuple[float, float]:
    '''Exact draw from N(mean,cov) conditional on c>0.

    Draw the truncated marginal for c, then beta from its conditional normal.
    This avoids the biased finite-retry fallback used by the earlier implementation.
    '''
    var_c = max(float(cov[0, 0]), 1e-18)
    sd_c = math.sqrt(var_c)
    lower_cdf = float(ndtr((0.0 - mean[0]) / sd_c))
    u = rng.uniform(min(lower_cdf, 1.0 - 1e-14), 1.0 - 1e-14)
    c = float(mean[0] + sd_c * ndtri(u))
    c = max(c, np.finfo(float).tiny)

    conditional_mean = float(mean[1] + cov[1, 0] / var_c * (c - mean[0]))
    conditional_var = max(float(cov[1, 1] - cov[1, 0] ** 2 / var_c), 1e-18)
    beta = float(rng.normal(conditional_mean, math.sqrt(conditional_var)))
    return c, beta


def update_q_checkerboard(
    q: np.ndarray,
    adjusted_delta_p: np.ndarray,
    c: float,
    sigma2: float,
    fixed_zero: np.ndarray,
    rng: np.random.Generator,
) -> None:
    '''Two-color Gibbs update for the latent direction path.

    Conditional on the opposite parity, q values of one parity are independent.
    Each candidate uses both adjacent disturbances, exactly as in Hasbrouck (2009).
    '''
    T = len(q)
    denom = 2.0 * max(float(sigma2), 1e-18)
    for parity in (0, 1):
        idx = np.arange(parity, T, 2, dtype=int)
        idx = idx[~fixed_zero[idx]]
        if len(idx) == 0:
            continue

        ss_plus = np.zeros(len(idx))
        ss_minus = np.zeros(len(idx))

        has_prev = idx > 0
        if has_prev.any():
            ii = idx[has_prev]
            y = adjusted_delta_p[ii - 1]
            q_prev = q[ii - 1]
            ss_plus[has_prev] += (y - c * (1.0 - q_prev)) ** 2
            ss_minus[has_prev] += (y - c * (-1.0 - q_prev)) ** 2

        has_next = idx < T - 1
        if has_next.any():
            ii = idx[has_next]
            y = adjusted_delta_p[ii]
            q_next = q[ii + 1]
            ss_plus[has_next] += (y - c * (q_next - 1.0)) ** 2
            ss_minus[has_next] += (y - c * (q_next + 1.0)) ** 2

        log_odds_plus = np.clip((ss_minus - ss_plus) / denom, -700, 700)
        p_plus = expit(log_odds_plus)
        q[idx] = np.where(rng.random(len(idx)) < p_plus, 1.0, -1.0)


def gibbs_hasbrouck(
    log_price: np.ndarray,
    market_log_level: np.ndarray,
    fixed_zero: np.ndarray | None = None,
    config: GibbsConfig = GibbsConfig(),
    seed: int = SEED,
    keep_draws: bool = False,
) -> dict:
    p = np.asarray(log_price, dtype=float)
    pm = np.asarray(market_log_level, dtype=float)
    if len(p) != len(pm):
        raise ValueError("price and market arrays must have equal length")
    if len(p) < 4 or config.burn_in >= config.n_sweeps:
        raise ValueError("insufficient observations or invalid burn-in")

    fixed_zero = np.zeros(len(p), dtype=bool) if fixed_zero is None else np.asarray(fixed_zero, dtype=bool)
    if len(fixed_zero) != len(p):
        raise ValueError("fixed_zero must match price length")

    dp = np.diff(p)
    dm = np.diff(pm)
    valid = np.isfinite(dp) & np.isfinite(dm)
    if not valid.all():
        raise ValueError("non-finite price or market increment")

    q = np.empty(len(p), dtype=float)
    q[0] = 1.0
    q[1:] = np.sign(dp)
    q[q == 0] = 1.0
    q[fixed_zero] = 0.0

    prior_mean = np.array([config.prior_c_mean, config.prior_beta_mean], dtype=float)
    prior_precision = np.diag([1.0 / config.prior_c_var, 1.0 / config.prior_beta_var])
    sigma2 = float(config.initial_sigma2)
    rng = np.random.default_rng(seed)

    c_draws = np.empty(config.n_sweeps)
    beta_draws = np.empty(config.n_sweeps)
    sigma2_draws = np.empty(config.n_sweeps)

    for sweep in range(config.n_sweeps):
        dq = np.diff(q)
        X = np.column_stack((dq, dm))
        posterior_precision = X.T @ X / sigma2 + prior_precision
        posterior_cov = np.linalg.inv(posterior_precision)
        posterior_mean = posterior_cov @ (X.T @ dp / sigma2 + prior_precision @ prior_mean)
        c, beta = draw_joint_normal_truncated_c(posterior_mean, posterior_cov, rng)

        residual = dp - c * dq - beta * dm
        shape = config.prior_sigma_alpha + len(dp) / 2.0
        scale = config.prior_sigma_scale + float(residual @ residual) / 2.0
        precision = rng.gamma(shape=shape, scale=1.0 / max(scale, 1e-30))
        sigma2 = max(1.0 / precision, 1e-18)

        adjusted = dp - beta * dm
        update_q_checkerboard(q, adjusted, c, sigma2, fixed_zero, rng)

        c_draws[sweep] = c
        beta_draws[sweep] = beta
        sigma2_draws[sweep] = sigma2

    c_post = c_draws[config.burn_in:]
    beta_post = beta_draws[config.burn_in:]
    sigma_post = sigma2_draws[config.burn_in:]
    result = {
        "c_gibbs": float(c_post.mean()),
        "c_gibbs_sd": float(c_post.std(ddof=1)),
        "c_gibbs_p05": float(np.quantile(c_post, 0.05)),
        "c_gibbs_p50": float(np.quantile(c_post, 0.50)),
        "c_gibbs_p95": float(np.quantile(c_post, 0.95)),
        "beta_m": float(beta_post.mean()),
        "beta_m_sd": float(beta_post.std(ddof=1)),
        "sigma2_u": float(sigma_post.mean()),
        "n_prices": len(p),
        "n_increments": len(dp),
    }
    if keep_draws:
        result["c_draws"] = c_post
        result["beta_draws"] = beta_post
    return result


## Unit validation: conditional odds, synthetic recovery, R-hat, and ESS


In [7]:
def q_loglik_bruteforce(q, adjusted_dp, c, sigma2, i, candidate):
    qc = q.copy()
    qc[i] = candidate
    residual = adjusted_dp - c * np.diff(qc)
    return -0.5 * float(residual @ residual) / sigma2


# Verify the local two-adjacent-residual calculation against the full path likelihood.
rng_check = np.random.default_rng(SEED)
q_check = rng_check.choice([-1.0, 1.0], size=17)
adjusted_check = rng_check.normal(0, 0.02, size=16)
c_check, s2_check, i_check = 0.013, 0.0002, 8
full_log_odds = (
    q_loglik_bruteforce(q_check, adjusted_check, c_check, s2_check, i_check, 1.0)
    - q_loglik_bruteforce(q_check, adjusted_check, c_check, s2_check, i_check, -1.0)
)
prev_y, next_y = adjusted_check[i_check - 1], adjusted_check[i_check]
local_plus = (prev_y - c_check * (1 - q_check[i_check - 1])) ** 2 + (next_y - c_check * (q_check[i_check + 1] - 1)) ** 2
local_minus = (prev_y - c_check * (-1 - q_check[i_check - 1])) ** 2 + (next_y - c_check * (q_check[i_check + 1] + 1)) ** 2
local_log_odds = (local_minus - local_plus) / (2 * s2_check)
assert np.isclose(full_log_odds, local_log_odds, atol=1e-12)


def simulate_roll_hasbrouck(T=500, c=0.012, beta=1.1, sigma_u=0.01, sigma_m=0.009, seed=1):
    rng = np.random.default_rng(seed)
    q = rng.choice([-1.0, 1.0], size=T)
    rm = rng.normal(0, sigma_m, size=T - 1)
    u = rng.normal(0, sigma_u, size=T - 1)
    m = np.r_[0.0, np.cumsum(beta * rm + u)]
    p = m + c * q
    pm = np.r_[0.0, np.cumsum(rm)]
    return p, pm, q


def split_rhat(chains: np.ndarray) -> float:
    chains = np.asarray(chains, float)
    m, n = chains.shape
    if n % 2:
        chains = chains[:, :-1]
        n -= 1
    split = np.concatenate([chains[:, : n // 2], chains[:, n // 2 :]], axis=0)
    n2 = split.shape[1]
    chain_means = split.mean(axis=1)
    B = n2 * chain_means.var(ddof=1)
    W = split.var(axis=1, ddof=1).mean()
    var_hat = (n2 - 1) / n2 * W + B / n2
    return float(np.sqrt(var_hat / W)) if W > 0 else np.nan


def effective_sample_size(chains: np.ndarray) -> float:
    chains = np.asarray(chains, float)
    m, n = chains.shape
    centered = chains - chains.mean(axis=1, keepdims=True)
    variances = np.sum(centered ** 2, axis=1)
    rho = []
    for lag in range(1, n):
        ac = np.mean([
            np.dot(x[:-lag], x[lag:]) / v if v > 0 else 0.0
            for x, v in zip(centered, variances)
        ])
        rho.append(ac)
        if lag % 2 == 0 and len(rho) >= 2 and rho[-1] + rho[-2] < 0:
            rho = rho[:-2]
            break
    tau = max(1.0, 1.0 + 2.0 * sum(rho))
    return float(min(m * n, m * n / tau))


p_sim, pm_sim, _ = simulate_roll_hasbrouck(seed=SEED)
validation_cfg = GibbsConfig(n_sweeps=1_400, burn_in=400)
validation_runs = [
    gibbs_hasbrouck(p_sim, pm_sim, config=validation_cfg, seed=SEED + k, keep_draws=True)
    for k in range(4)
]
c_chains = np.stack([run["c_draws"] for run in validation_runs])
synthetic_validation = pd.DataFrame([{
    "true_c": 0.012,
    "posterior_c_mean": float(c_chains.mean()),
    "absolute_error": float(abs(c_chains.mean() - 0.012)),
    "split_rhat": split_rhat(c_chains),
    "ess": effective_sample_size(c_chains),
    "mcse": float(c_chains.std(ddof=1) / math.sqrt(effective_sample_size(c_chains))),
    "roll_c": roll_estimate(np.diff(p_sim))["c_roll"],
    "conditional_odds_identity_passed": True,
}])
synthetic_validation.to_csv(OUTPUT_DIR / "synthetic_validation.csv", index=False)
print(synthetic_validation.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


  true_c  posterior_c_mean  absolute_error  split_rhat         ess     mcse   roll_c  conditional_odds_identity_passed
0.012000          0.012061        0.000061    1.000408 1741.958572 0.000009 0.010625                              True


## Estimate the selected SCDLDS firm/portfolio-year samples


In [ ]:
def estimate_group(group: pd.DataFrame, seed: int, config: GibbsConfig = GibbsConfig()) -> dict:
    g = group.sort_values("date").drop_duplicates("date", keep="last")
    log_price = np.log(g["close"].to_numpy(float))
    market_level = g["market_log_level"].to_numpy(float)

    # No observed volume==0 rows exist in this panel. Missing volume is unknown,
    # not proof of a no-trade midpoint, so it is not fixed to q=0.
    fixed_zero = g["volume"].eq(0).fillna(False).to_numpy(bool)
    dp = np.diff(log_price)
    roll = roll_estimate(dp)
    gibbs = gibbs_hasbrouck(log_price, market_level, fixed_zero=fixed_zero, config=config, seed=seed, keep_draws=True)
    c_draws = gibbs.pop("c_draws")
    gibbs.pop("beta_draws")
    gibbs["c_chain_ess"] = effective_sample_size(c_draws[None, :])
    gibbs["c_chain_mcse"] = float(np.std(c_draws, ddof=1) / math.sqrt(gibbs["c_chain_ess"]))

    simple_ret = np.expm1(dp)
    rupee_volume = (g["close"].to_numpy(float)[1:] * g["volume"].to_numpy(float)[1:])
    valid_dv = np.isfinite(rupee_volume) & (rupee_volume > 0)
    amihud = float(np.mean(np.abs(simple_ret[valid_dv]) / rupee_volume[valid_dv])) if valid_dv.any() else np.nan

    result = {
        "company_code": int(g["company_code"].iloc[0]),
        "sample_portfolio_year": int(g["portfolio_year"].iloc[0]),
        "first_date": str(g["date"].iloc[0].date()),
        "last_date": str(g["date"].iloc[-1].date()),
        "avg_mktcap": float(g["market_cap"].mean()),
        "prop_zero_price_change": float(np.mean(np.isclose(dp, 0.0, atol=1e-12))),
        "amihud_raw": amihud,
        **roll,
        **gibbs,
    }
    result["one_way_cost_bps"] = 10_000.0 * result["c_gibbs"]
    result["effective_spread_bps"] = 20_000.0 * result["c_gibbs"]
    result["roll_one_way_cost_bps"] = 10_000.0 * result["c_roll"]
    return result


grouped = daily.groupby(["company_code", "portfolio_year"], observed=True, sort=True)
rows = []
t0 = time.time()
selected_total = len(selected_keys)
report_every = max(1, selected_total // 10)
done = 0
for (company_code, portfolio_year), group in grouped:
    key = (int(company_code), int(portfolio_year))
    if key not in selected_keys:
        continue
    seed = int((SEED + key[0] * 1009 + key[1] * 9176) % (2**32 - 1))
    result = estimate_group(group, seed)
    relative_mcse = result["c_chain_mcse"] / result["c_gibbs"]
    needs_longer_chain = result["c_chain_ess"] < MIN_CHAIN_ESS or relative_mcse > MAX_RELATIVE_MCSE
    if needs_longer_chain:
        longer = GibbsConfig(n_sweeps=ADAPTIVE_N_SWEEPS, burn_in=ADAPTIVE_BURN_IN)
        result = estimate_group(group, seed + 7_919, config=longer)
    result["adaptive_rerun"] = bool(needs_longer_chain)
    result["n_sweeps_used"] = ADAPTIVE_N_SWEEPS if needs_longer_chain else N_SWEEPS
    result["relative_mcse"] = result["c_chain_mcse"] / result["c_gibbs"]
    rows.append(result)
    done += 1
    if done % report_every == 0 or done == selected_total:
        print(f"estimated {done}/{selected_total} groups in {time.time()-t0:.1f}s")

estimates = pd.DataFrame(rows).sort_values(["sample_portfolio_year", "company_code"]).reset_index(drop=True)
result_name = "transaction_cost_estimates_full.parquet" if RUN_MODE == "full" else "transaction_cost_estimates_quick.parquet"
estimates.to_parquet(OUTPUT_DIR / result_name, index=False)
print(f"Saved {len(estimates):,} estimates to {OUTPUT_DIR / result_name}")


## Results and internal empirical validation


In [ ]:
summary_columns = [
    "one_way_cost_bps", "effective_spread_bps", "roll_one_way_cost_bps",
    "beta_m", "c_chain_ess", "c_chain_mcse", "relative_mcse", "prop_zero_price_change", "amihud_raw",
]
summary = estimates[summary_columns].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
summary.to_csv(OUTPUT_DIR / f"summary_{RUN_MODE}.csv")
print(summary.to_string(float_format=lambda x: f"{x:.6g}"))

feasible = estimates.loc[estimates["roll_feasible"] & estimates["c_roll"].notna()].copy()
validation_rows = []

def safe_corr(x, y, method):
    pair = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(pair) < 3 or pair["x"].nunique() < 2 or pair["y"].nunique() < 2:
        return np.nan, np.nan, len(pair)
    stat, pvalue = (spearmanr(pair["x"], pair["y"]) if method == "spearman" else pearsonr(pair["x"], pair["y"]))
    return float(stat), float(pvalue), len(pair)

for name, x, y, expected in [
    ("Gibbs vs feasible Roll", feasible["c_gibbs"], feasible["c_roll"], "positive"),
    ("Gibbs vs log market cap", estimates["c_gibbs"], np.log(estimates["avg_mktcap"]), "negative"),
    ("Gibbs vs zero-price-change share", estimates["c_gibbs"], estimates["prop_zero_price_change"], "positive"),
    ("Gibbs vs Amihud", estimates["c_gibbs"], estimates["amihud_raw"], "positive"),
]:
    for method in ("spearman", "pearson"):
        stat, pvalue, n = safe_corr(x, y, method)
        validation_rows.append({"comparison": name, "method": method, "correlation": stat, "p_value": pvalue, "n": n, "expected_sign": expected})

empirical_validation = pd.DataFrame(validation_rows)
empirical_validation.to_csv(OUTPUT_DIR / f"empirical_validation_{RUN_MODE}.csv", index=False)
print("\nEmpirical validation correlations")
print(empirical_validation.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Published values are context, not calibration targets: India, sample composition,
# tick sizes, and market structure differ. Hasbrouck Table II reports mean/median c
# of 0.0112/0.0061 in the 1993-2005 U.S. comparison sample.
print("\nContext check:")
print({
    "sample_median_c": float(estimates["c_gibbs"].median()),
    "sample_mean_c": float(estimates["c_gibbs"].mean()),
    "hasbrouck_us_table_II_median_c": 0.0061,
    "hasbrouck_us_table_II_mean_c": 0.0112,
    "roll_infeasible_share": float((~estimates["roll_feasible"]).mean()),
    "adaptive_long_chain_share": float(estimates["adaptive_rerun"].mean()),
    "minimum_final_ess": float(estimates["c_chain_ess"].min()),
    "maximum_final_relative_mcse": float(estimates["relative_mcse"].max()),
})


### Multi-chain MCMC audit on representative empirical firm-years


In [ ]:
# Single-chain ESS/MCSE is stored for every estimate. For a stronger mixing check,
# rerun three empirical groups (low/median/high estimated cost) with four chains.
ordered = estimates.sort_values("c_gibbs").reset_index(drop=True)
audit_positions = np.unique(np.round(np.array([0.10, 0.50, 0.90]) * (len(ordered) - 1)).astype(int))
mcmc_rows = []
audit_cfg = GibbsConfig(n_sweeps=ADAPTIVE_N_SWEEPS, burn_in=ADAPTIVE_BURN_IN)
for pos in audit_positions:
    row = ordered.iloc[int(pos)]
    key = (int(row["company_code"]), int(row["sample_portfolio_year"]))
    g = grouped.get_group(key).sort_values("date").drop_duplicates("date", keep="last")
    lp = np.log(g["close"].to_numpy(float))
    ml = g["market_log_level"].to_numpy(float)
    fz = g["volume"].eq(0).fillna(False).to_numpy(bool)
    chains = []
    for chain_id in range(4):
        run = gibbs_hasbrouck(
            lp, ml, fixed_zero=fz, config=audit_cfg,
            seed=int((SEED + key[0] * 1009 + key[1] * 9176 + chain_id * 104729) % (2**32 - 1)),
            keep_draws=True,
        )
        chains.append(run["c_draws"])
    chains = np.stack(chains)
    ess = effective_sample_size(chains)
    mcse = float(chains.std(ddof=1) / math.sqrt(ess))
    rhat = split_rhat(chains)
    mcmc_rows.append({
        "company_code": key[0],
        "sample_portfolio_year": key[1],
        "cost_quantile_target": float([0.10, 0.50, 0.90][len(mcmc_rows)]),
        "four_chain_c_mean": float(chains.mean()),
        "split_rhat": rhat,
        "ess": ess,
        "mcse": mcse,
        "relative_mcse": mcse / float(chains.mean()),
        "passes_rhat_1_01": bool(rhat < 1.01),
    })

empirical_mcmc_audit = pd.DataFrame(mcmc_rows)
empirical_mcmc_audit.to_csv(OUTPUT_DIR / f"empirical_mcmc_audit_{RUN_MODE}.csv", index=False)
print(empirical_mcmc_audit.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


In [ ]:
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].hist(estimates["one_way_cost_bps"].clip(upper=estimates["one_way_cost_bps"].quantile(0.99)), bins=30, color="#276FBF", alpha=0.85)
axes[0].set(title="Hasbrouck one-way cost", xlabel="basis points (winsorized for display)", ylabel="firm-years")

positive_size = estimates["avg_mktcap"] > 0
axes[1].scatter(np.log(estimates.loc[positive_size, "avg_mktcap"]), estimates.loc[positive_size, "one_way_cost_bps"], s=13, alpha=0.45, color="#E4572E")
axes[1].set(title="Cost versus firm size", xlabel="log average market cap", ylabel="one-way cost (bps)")

annual = estimates.groupby("sample_portfolio_year")["one_way_cost_bps"].median()
axes[2].plot(annual.index, annual.values, marker="o", color="#2E933C")
axes[2].set(title="Cross-sectional median by portfolio year", xlabel="sample portfolio year", ylabel="one-way cost (bps)")
axes[2].xaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
axes[2].grid(alpha=0.2)

fig.tight_layout()
figure_path = OUTPUT_DIR / f"transaction_cost_diagnostics_{RUN_MODE}.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.close(fig)
print(f"Saved {figure_path}")


## Look-ahead-safe website merge schedule


In [ ]:
schedule = estimates[[
    "company_code", "sample_portfolio_year", "one_way_cost_bps", "effective_spread_bps",
    "c_gibbs", "c_gibbs_sd", "c_gibbs_p05", "c_gibbs_p95", "n_prices",
]].copy()
schedule["apply_portfolio_year"] = schedule["sample_portfolio_year"] + 1
schedule["estimator"] = "Hasbrouck2009_market_factor_Gibbs"
schedule["is_lookahead_safe"] = True

schedule_name = f"website_cost_schedule_{RUN_MODE}.csv"
schedule.to_csv(OUTPUT_DIR / schedule_name, index=False)

coverage = website[["co_code", "Month", "portfolio_year"]].merge(
    schedule[["company_code", "apply_portfolio_year", "one_way_cost_bps"]],
    left_on=["co_code", "portfolio_year"],
    right_on=["company_code", "apply_portfolio_year"],
    how="left",
)
coverage_by_year = coverage.groupby("portfolio_year")["one_way_cost_bps"].agg(rows="size", matched="count")
coverage_by_year["coverage"] = coverage_by_year["matched"] / coverage_by_year["rows"]
coverage_by_year.to_csv(OUTPUT_DIR / f"website_coverage_{RUN_MODE}.csv")

print(f"Saved {OUTPUT_DIR / schedule_name}")
print(coverage_by_year.tail(10).to_string(float_format=lambda x: f"{x:.3f}"))
print("Quick-mode coverage is intentionally sparse; run TCOST_RUN_MODE=full before website integration.")


## External verification with intraday quotes or broker executions

Internal diagnostics can catch coding errors and implausible behavior, but they
cannot prove that daily-data estimates equal executable costs. The strongest
verification mirrors Hasbrouck’s paper: obtain a matched intraday sample with
execution price and the prevailing bid/ask midpoint, compute
`abs(log(execution_price / midpoint))`, dollar-weight it within each firm-year,
and compare it with `c_gibbs` using Pearson/Spearman correlations, calibration
slope/intercept, MAE, and coverage of posterior intervals.

The function below is ready for such a file. It intentionally refuses to infer a
quote midpoint from OHLC bars.


In [ ]:
def validate_against_intraday_quotes(est: pd.DataFrame, trades: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    required = {"company_code", "date", "execution_price", "midpoint", "rupee_value"}
    missing = required - set(trades.columns)
    if missing:
        raise ValueError(f"intraday trade file is missing columns: {sorted(missing)}")

    q = trades.copy()
    q["date"] = pd.to_datetime(q["date"])
    valid = (
        q["execution_price"].gt(0)
        & q["midpoint"].gt(0)
        & q["rupee_value"].gt(0)
    )
    q = q.loc[valid].copy()
    q["sample_portfolio_year"] = portfolio_year_from_date(q["date"]).astype(int)
    q["realized_effective_cost"] = np.abs(np.log(q["execution_price"] / q["midpoint"]))
    q["weighted_cost"] = q["realized_effective_cost"] * q["rupee_value"]
    realized = (
        q.groupby(["company_code", "sample_portfolio_year"], observed=True)
        .agg(weighted_cost=("weighted_cost", "sum"), rupee_value=("rupee_value", "sum"), n_trades=("realized_effective_cost", "size"))
        .reset_index()
    )
    realized["c_intraday"] = realized["weighted_cost"] / realized["rupee_value"]
    matched = est.merge(realized, on=["company_code", "sample_portfolio_year"], how="inner")
    if len(matched) < 3:
        return matched, {"n": len(matched), "pearson": np.nan, "spearman": np.nan, "mae": np.nan}
    diagnostics = {
        "n": len(matched),
        "pearson": float(pearsonr(matched["c_gibbs"], matched["c_intraday"]).statistic),
        "spearman": float(spearmanr(matched["c_gibbs"], matched["c_intraday"]).statistic),
        "mae": float(np.mean(np.abs(matched["c_gibbs"] - matched["c_intraday"]))),
        "posterior_90pct_coverage": float(np.mean(matched["c_intraday"].between(matched["c_gibbs_p05"], matched["c_gibbs_p95"]))),
    }
    return matched, diagnostics

print("Intraday verifier defined; not run because no trade-and-quote file was supplied.")


## Conclusions and next integration step

- The notebook implements the paper’s actual latent-direction model and correct
  adjacent-residual Gibbs conditionals.
- Website panel membership, not a re-created approximation, controls the investable
  universe; the explicit filter reconstruction is retained as an audit.
- Estimates are annual/portfolio-year, and the exported application year is lagged
  to avoid look-ahead.
- `one_way_cost_bps` is the field the backtest should ultimately charge against
  absolute weight traded. `effective_spread_bps` is doubled and is for reporting.
- A full run is required before website integration. The current `backtest-core.js`
  accepts one scalar bps input; firm-specific costs will require the turnover loop to
  multiply each security’s absolute weight change by its own lagged cost.
- Definitive economic validation requires intraday quotes or broker execution data.
  Synthetic recovery, Roll/Amihud/size relations, MCMC diagnostics, and filter audits
  are necessary checks, but not substitutes for that external benchmark.
